Uso le **197 anteprime audio**, una per speaker del canale scelto, per rendere più rapida l'identificazione dei ruoli.

Obiettivi:

1. trascrivere localmente ogni anteprima con Faster-Whisper;
2. affiancare testo, durata e priorità di revisione;
3. proporre un ruolo preliminare tramite semplici indicatori linguistici;
4. permettere la revisione interattiva con ascolto e menu a tendina;
5. salvare una mappa finale `registrazione–speaker–ruolo`.

Ruoli disponibili:

- `paziente`
- `intervistatore`
- `voce_registrata`
- `terza_persona`
- `rumore`
- `incerto`

> Il ruolo suggerito è soltanto un supporto. Non deve essere considerato una ground truth senza controllo.


In [5]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import Audio, HTML, display

PROJECT_DIR = Path.cwd()
SELECTION_DIR = PROJECT_DIR / "risultati" / "selezione_canale"
PREVIEW_DIR = SELECTION_DIR / "anteprime_speaker"

QUEUE_CSV = SELECTION_DIR / "coda_revisione_speaker.csv"
TRANSCRIPT_CSV = SELECTION_DIR / "trascrizioni_anteprime.csv"
REVIEW_CSV = SELECTION_DIR / "mappa_ruoli_speaker_da_revisionare.csv"
FINAL_CSV = SELECTION_DIR / "mappa_ruoli_speaker_finale.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("QUEUE_CSV esiste:", QUEUE_CSV.exists())
print("PREVIEW_DIR esiste:", PREVIEW_DIR.exists())

if not QUEUE_CSV.exists():
    raise FileNotFoundError(
        f"File non trovato: {QUEUE_CSV}\n"
        "Eseguire prima il notebook 05."
    )


PROJECT_DIR: c:\Users\acer\Desktop\ProgettoTesi
QUEUE_CSV esiste: True
PREVIEW_DIR esiste: True


## 1. Caricamento della coda

Sono attese:

- 93 registrazioni;
- 197 speaker sul canale scelto;
- 10 registrazioni con più di 2 speaker;
- 5 registrazioni con discordanza tra i canali.


In [6]:
coda = pd.read_csv(QUEUE_CSV)

colonne_richieste = {
    "recording_id", "canale", "speaker",
    "priorita_revisione", "percorso_anteprima",
    "durata_totale_secondi",
    "quota_parlato_nella_registrazione"
}

mancanti = colonne_richieste - set(coda.columns)
if mancanti:
    raise ValueError(f"Colonne mancanti: {sorted(mancanti)}")

print("Righe speaker:", len(coda))
print("Registrazioni:", coda["recording_id"].nunique())
print("\nPriorità:")
display(
    coda["priorita_revisione"]
    .value_counts()
    .rename_axis("priorita")
    .to_frame("speaker")
)

display(coda.head())


Righe speaker: 197
Registrazioni: 93

Priorità:


,speaker
priorita,
ordinaria,156
alta: più di 2 speaker,31
media: canali discordanti,10


,patient_id,recording_id,nome_file,canale,speaker,numero_segmenti,durata_totale_secondi,durata_media_secondi,durata_massima_secondi,quota_parlato_nella_registrazione,...,numero_speaker_canale_scelto,discordanza_numero_speaker,margine_punteggio,priorita_revisione,ruolo_manual,confidenza_manual,note_manual,percorso_anteprima,nome_anteprima,durata_anteprima_secondi
0,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,SPEAKER_00,34,71.615,2.106324,5.923,0.252582,...,2,False,4.0,ordinaria,NaN,NaN,NaN,c:\Users\acer\Desktop\ProgettoTesi\risultati\s...,ID1.StudioRuggi-006__destro__SPEAKER_00.wav,26.25
1,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006.wav,destro,SPEAKER_01,39,211.917,5.433769,26.258,0.747418,...,2,False,4.0,ordinaria,NaN,NaN,NaN,c:\Users\acer\Desktop\ProgettoTesi\risultati\s...,ID1.StudioRuggi-006__destro__SPEAKER_01.wav,25.50
2,2,ID2.StudioRuggi,ID2.StudioRuggi.wav,sinistro,SPEAKER_00,50,81.929,1.638580,7.594,0.507263,...,2,False,2.0,ordinaria,NaN,NaN,NaN,c:\Users\acer\Desktop\ProgettoTesi\risultati\s...,ID2.StudioRuggi__sinistro__SPEAKER_00.wav,25.75
3,2,ID2.StudioRuggi,ID2.StudioRuggi.wav,sinistro,SPEAKER_01,42,79.583,1.894833,9.011,0.492737,...,2,False,2.0,ordinaria,NaN,NaN,NaN,c:\Users\acer\Desktop\ProgettoTesi\risultati\s...,ID2.StudioRuggi__sinistro__SPEAKER_01.wav,25.75
4,3,ID3.StudioRuggi,ID3.StudioRuggi.wav,sinistro,SPEAKER_00,29,63.888,2.203034,5.417,0.338724,...,2,False,2.0,ordinaria,NaN,NaN,NaN,c:\Users\acer\Desktop\ProgettoTesi\risultati\s...,ID3.StudioRuggi__sinistro__SPEAKER_00.wav,26.50


## 2. Configurazione di Faster-Whisper

Configurazione predefinita:

- GPU NVIDIA disponibile: `large-v3-turbo`, `cuda`, `float16`;
- solo CPU: `small`, `cpu`, `int8`.

Il modello `small` è scelto su CPU per ridurre i tempi.  
Per una trascrizione più accurata è possibile sostituirlo con `medium`.


In [7]:
try:
    import torch
    CUDA_DISPONIBILE = torch.cuda.is_available()
except Exception:
    CUDA_DISPONIBILE = False

if CUDA_DISPONIBILE:
    MODEL_SIZE = "large-v3-turbo"
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
else:
    MODEL_SIZE = "small"
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"

BEAM_SIZE = 5
LINGUA = "it"

print("CUDA disponibile:", CUDA_DISPONIBILE)
print("MODEL_SIZE:", MODEL_SIZE)
print("DEVICE:", DEVICE)
print("COMPUTE_TYPE:", COMPUTE_TYPE)


CUDA disponibile: False
MODEL_SIZE: small
DEVICE: cpu
COMPUTE_TYPE: int8


In [5]:
%pip install -U faster-whisper ipywidgets

Note: you may need to restart the kernel to use updated packages.Collecting faster-whisper
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 9.1 MB/s  0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   --- ------------------------------------ 1.8/19.2 MB 8.4 MB/s eta 0:00:03
   --------- ------------------------------ 4.7/19.2 MB 11.9 MB/s eta 0:00:02
   -------------- ------------------------- 7.1/19.2 MB 11.8 MB/s eta 0:00:02
   -------------------- ------------------- 9.7/19.2 MB 11.8 MB/s eta 0:00:01
   ------------------------- -------------- 12.1/19.2 MB 11.8 MB/s eta 0:00:01
   ------------------------------ --------- 14.4/19.2 MB 11.9 MB/s eta 0:00:01
   ----------------------------------- ---- 17.0/19.2 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  19.1/19.2 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 19.2/19.2 MB 11.6 MB/s  0:

Impossibile trovare il percorso specificato.


## 3. Trascrizione batch con ripresa automatica

Il CSV viene aggiornato dopo ogni anteprima.  
In caso di interruzione, rilanciando la cella verranno saltati i file già completati.


In [8]:
try:
    from faster_whisper import WhisperModel
except ImportError as exc:
    raise ImportError(
        "faster-whisper non è installato. "
        "Eseguire: %pip install faster-whisper"
    ) from exc


if TRANSCRIPT_CSV.exists():
    trascrizioni_esistenti = pd.read_csv(TRANSCRIPT_CSV)
else:
    trascrizioni_esistenti = pd.DataFrame(
        columns=[
            "nome_anteprima", "recording_id", "canale",
            "speaker", "trascrizione", "lingua_rilevata",
            "probabilita_lingua", "stato_trascrizione", "errore"
        ]
    )

gia_completati = set(
    trascrizioni_esistenti.loc[
        trascrizioni_esistenti["stato_trascrizione"] == "completato",
        "nome_anteprima"
    ].astype(str)
)

print("Anteprime già trascritte:", len(gia_completati))
print("Caricamento del modello...")

modello = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)

risultati = trascrizioni_esistenti.to_dict("records")

for indice, riga in coda.reset_index(drop=True).iterrows():
    percorso = Path(str(riga["percorso_anteprima"]))
    nome = percorso.name

    if nome in gia_completati:
        print(
            f"\r[{indice + 1}/{len(coda)}] "
            f"{nome}: già completato, salto.",
            end=""
        )
        continue

    print(
        f"\r[{indice + 1}/{len(coda)}] "
        f"Trascrizione: {nome}",
        end=""
    )

    record = {
        "nome_anteprima": nome,
        "recording_id": riga["recording_id"],
        "canale": riga["canale"],
        "speaker": riga["speaker"],
        "trascrizione": "",
        "lingua_rilevata": "",
        "probabilita_lingua": np.nan,
        "stato_trascrizione": "errore",
        "errore": "",
    }

    try:
        if not percorso.exists():
            raise FileNotFoundError(percorso)

        segmenti, info = modello.transcribe(
            str(percorso),
            language=LINGUA,
            beam_size=BEAM_SIZE,
            vad_filter=True,
            condition_on_previous_text=False,
        )

        parti_testo = []
        for segmento in segmenti:
            testo = segmento.text.strip()
            if testo:
                parti_testo.append(testo)

        record["trascrizione"] = " ".join(parti_testo).strip()
        record["lingua_rilevata"] = getattr(info, "language", "")
        record["probabilita_lingua"] = getattr(
            info, "language_probability", np.nan
        )
        record["stato_trascrizione"] = "completato"

    except Exception as exc:
        record["errore"] = str(exc)

    # Sostituisce un eventuale record precedente dello stesso file
    risultati = [
        x for x in risultati
        if str(x.get("nome_anteprima")) != nome
    ]
    risultati.append(record)

    pd.DataFrame(risultati).to_csv(
        TRANSCRIPT_CSV,
        index=False
    )

print("\nTrascrizione batch terminata.")

trascrizioni = pd.read_csv(TRANSCRIPT_CSV)

print("\nStati:")
display(
    trascrizioni["stato_trascrizione"]
    .value_counts(dropna=False)
    .rename_axis("stato")
    .to_frame("conteggio")
)

print("Salvato in:", TRANSCRIPT_CSV)


Anteprime già trascritte: 197
Caricamento del modello...
[197/197] ID164.StudioRuggi__sinistro__SPEAKER_01.wav: già completato, salto..o.
Trascrizione batch terminata.

Stati:


,conteggio
stato,
completato,197


Salvato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\selezione_canale\trascrizioni_anteprime.csv


## 4. Unione tra dati di diarizzazione e trascrizioni


In [9]:
dati = coda.merge(
    trascrizioni[
        [
            "nome_anteprima", "trascrizione",
            "lingua_rilevata", "probabilita_lingua",
            "stato_trascrizione", "errore"
        ]
    ],
    on="nome_anteprima",
    how="left"
)

dati["trascrizione"] = dati["trascrizione"].fillna("")
dati["numero_caratteri"] = dati["trascrizione"].str.len()
dati["numero_parole"] = (
    dati["trascrizione"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

display(
    dati[
        [
            "recording_id", "speaker",
            "priorita_revisione",
            "numero_parole", "trascrizione"
        ]
    ].head(20)
)


,recording_id,speaker,priorita_revisione,numero_parole,trascrizione
0,ID1.StudioRuggi-006,SPEAKER_00,ordinaria,72,Se dovessi descrivere questo dolore attraverso...
1,ID1.StudioRuggi-006,SPEAKER_01,ordinaria,60,"Beh, mi ricordo di mare scenate, pure brutte, ..."
2,ID2.StudioRuggi,SPEAKER_00,ordinaria,41,"si crescevano bene, senza malattimi, senza sol..."
3,ID2.StudioRuggi,SPEAKER_01,ordinaria,65,"E si ricorda come erano i svilviti da piccoli,..."
4,ID3.StudioRuggi,SPEAKER_00,ordinaria,73,riuscirebbe ad escriverlo quindi come un simbo...
5,ID3.StudioRuggi,SPEAKER_01,ordinaria,47,"Questo è una cosa che mi fa molto male, perché..."
6,ID4.StudioRuggi,SPEAKER_00,ordinaria,75,Può provare a descrivermi l'episodio in cui si...
7,ID4.StudioRuggi,SPEAKER_01,ordinaria,75,il maggio dell'anno scorso è passato un anno e...
8,ID10.StudioRuggi,SPEAKER_00,ordinaria,78,"si sente compreso da questa persona, quando le..."
9,ID10.StudioRuggi,SPEAKER_01,ordinaria,58,forse questo perché poi oltre a questo dolore ...


## 5. Suggerimento linguistico del ruolo

Il suggerimento usa segnali molto semplici:

- domande, seconda persona e formule cliniche → probabile intervistatore;
- prima persona, sintomi, terapie ed esperienza personale → probabile paziente.

Non vengono usati sesso della voce, durata o speaker dominante per assegnare il ruolo.


In [10]:
CUE_INTERVISTATORE = [
    r"\bquanto\b",
    r"\bdove\b",
    r"\bquando\b",
    r"\bcome\b",
    r"\bperché\b",
    r"\bda quanto\b",
    r"\bmi dica\b",
    r"\bmi racconti\b",
    r"\bmi può\b",
    r"\bpuò descrivere\b",
    r"\bha dolore\b",
    r"\bsente\b",
    r"\ble capita\b",
    r"\bprende farmaci\b",
    r"\bda zero a dieci\b",
    r"\bscala\b",
    r"\bche tipo di\b",
    r"\bha mai\b",
    r"\bsoffre di\b",
]

CUE_PAZIENTE = [
    r"\bio\b",
    r"\bho\b",
    r"\bavevo\b",
    r"\bsono\b",
    r"\bsento\b",
    r"\bsentivo\b",
    r"\bmi fa\b",
    r"\bmi prende\b",
    r"\bmi viene\b",
    r"\bmi sento\b",
    r"\bnon riesco\b",
    r"\bprendo\b",
    r"\bprendevo\b",
    r"\bsono stato\b",
    r"\bsono stata\b",
    r"\bil mio dolore\b",
    r"\bbruciore\b",
    r"\bformicolio\b",
    r"\bscossa\b",
    r"\bfitte\b",
    r"\bdolore\b",
]


def normalizza_testo(testo):
    testo = str(testo or "").lower()
    testo = unicodedata.normalize("NFKD", testo)
    testo = "".join(
        carattere for carattere in testo
        if not unicodedata.combining(carattere)
    )
    testo = re.sub(r"\s+", " ", testo).strip()
    return testo


def conta_cue(testo, pattern):
    return sum(
        len(re.findall(p, testo, flags=re.IGNORECASE))
        for p in pattern
    )


def calcola_punteggi_ruolo(testo):
    testo_norm = normalizza_testo(testo)

    score_intervistatore = conta_cue(
        testo_norm,
        CUE_INTERVISTATORE
    )
    score_paziente = conta_cue(
        testo_norm,
        CUE_PAZIENTE
    )

    # Una domanda esplicita è un indizio aggiuntivo.
    score_intervistatore += str(testo).count("?")

    return score_paziente, score_intervistatore


punteggi = dati["trascrizione"].apply(calcola_punteggi_ruolo)

dati["score_paziente"] = [x[0] for x in punteggi]
dati["score_intervistatore"] = [x[1] for x in punteggi]
dati["differenza_ruolo"] = (
    dati["score_paziente"]
    - dati["score_intervistatore"]
)

dati["ruolo_suggerito"] = np.select(
    [
        dati["differenza_ruolo"] >= 2,
        dati["differenza_ruolo"] <= -2,
    ],
    [
        "paziente",
        "intervistatore_o_voce_registrata",
    ],
    default="incerto"
)

dati["confidenza_suggerimento"] = np.select(
    [
        dati["differenza_ruolo"].abs() >= 5,
        dati["differenza_ruolo"].abs() >= 2,
    ],
    [
        "alta",
        "media",
    ],
    default="bassa"
)


### Vincolo relativo per le registrazioni con due speaker

Quando una registrazione ha esattamente due speaker, il notebook confronta i due punteggi.  
Quello con maggiore evidenza autobiografica viene proposto come paziente; l'altro come intervistatore/voce registrata.

La proposta resta da validare.


In [11]:
dati["numero_speaker_registrazione"] = (
    dati.groupby("recording_id")["speaker"]
    .transform("nunique")
)

for recording_id, indici in dati.groupby("recording_id").groups.items():
    indici = list(indici)
    sottoinsieme = dati.loc[indici]

    if sottoinsieme["speaker"].nunique() != 2:
        continue

    ordinati = sottoinsieme.sort_values(
        [
            "differenza_ruolo",
            "score_paziente",
            "numero_parole"
        ],
        ascending=[False, False, False]
    )

    indice_paziente = ordinati.index[0]
    indice_altro = ordinati.index[1]

    margine = (
        dati.at[indice_paziente, "differenza_ruolo"]
        - dati.at[indice_altro, "differenza_ruolo"]
    )

    dati.at[indice_paziente, "ruolo_suggerito"] = "paziente"
    dati.at[indice_altro, "ruolo_suggerito"] = (
        "intervistatore_o_voce_registrata"
    )

    if margine >= 5:
        confidenza = "alta"
    elif margine >= 2:
        confidenza = "media"
    else:
        confidenza = "bassa"

    dati.at[indice_paziente, "confidenza_suggerimento"] = confidenza
    dati.at[indice_altro, "confidenza_suggerimento"] = confidenza


dati["revisione_obbligatoria"] = (
    (dati["numero_speaker_registrazione"] != 2)
    | dati["priorita_revisione"].astype(str).str.startswith("alta")
    | dati["priorita_revisione"].astype(str).str.startswith("media")
    | (dati["confidenza_suggerimento"] == "bassa")
    | (dati["numero_parole"] < 3)
)

display(
    dati[
        [
            "recording_id", "speaker",
            "numero_speaker_registrazione",
            "priorita_revisione",
            "trascrizione",
            "score_paziente",
            "score_intervistatore",
            "ruolo_suggerito",
            "confidenza_suggerimento",
            "revisione_obbligatoria"
        ]
    ].head(30)
)


,recording_id,speaker,numero_speaker_registrazione,priorita_revisione,trascrizione,score_paziente,score_intervistatore,ruolo_suggerito,confidenza_suggerimento,revisione_obbligatoria
0,ID1.StudioRuggi-006,SPEAKER_00,2,ordinaria,Se dovessi descrivere questo dolore attraverso...,3,7,intervistatore_o_voce_registrata,alta,False
1,ID1.StudioRuggi-006,SPEAKER_01,2,ordinaria,"Beh, mi ricordo di mare scenate, pure brutte, ...",1,0,paziente,alta,False
2,ID2.StudioRuggi,SPEAKER_00,2,ordinaria,"si crescevano bene, senza malattimi, senza sol...",0,4,paziente,bassa,True
3,ID2.StudioRuggi,SPEAKER_01,2,ordinaria,"E si ricorda come erano i svilviti da piccoli,...",0,5,intervistatore_o_voce_registrata,bassa,True
4,ID3.StudioRuggi,SPEAKER_00,2,ordinaria,riuscirebbe ad escriverlo quindi come un simbo...,1,4,intervistatore_o_voce_registrata,media,False
5,ID3.StudioRuggi,SPEAKER_01,2,ordinaria,"Questo è una cosa che mi fa molto male, perché...",2,2,paziente,media,False
6,ID4.StudioRuggi,SPEAKER_00,2,ordinaria,Può provare a descrivermi l'episodio in cui si...,4,5,intervistatore_o_voce_registrata,media,False
7,ID4.StudioRuggi,SPEAKER_01,2,ordinaria,il maggio dell'anno scorso è passato un anno e...,4,1,paziente,media,False
8,ID10.StudioRuggi,SPEAKER_00,2,ordinaria,"si sente compreso da questa persona, quando le...",3,7,intervistatore_o_voce_registrata,alta,False
9,ID10.StudioRuggi,SPEAKER_01,2,ordinaria,forse questo perché poi oltre a questo dolore ...,6,0,paziente,alta,False


## 6. Creazione del file di revisione

La prima esecuzione crea il file.  
Le esecuzioni successive non sovrascrivono eventuali ruoli già compilati.


In [12]:
colonne_output = [
    "recording_id", "canale", "speaker",
    "numero_speaker_registrazione",
    "priorita_revisione",
    "durata_totale_secondi",
    "quota_parlato_nella_registrazione",
    "speaker_dominante",
    "nome_anteprima", "percorso_anteprima",
    "trascrizione", "numero_parole",
    "score_paziente", "score_intervistatore",
    "ruolo_suggerito", "confidenza_suggerimento",
    "revisione_obbligatoria"
]

revisione_nuova = dati[colonne_output].copy()
revisione_nuova["ruolo_manual"] = ""
revisione_nuova["confidenza_manual"] = ""
revisione_nuova["note_manual"] = ""

if REVIEW_CSV.exists():
    revisione_precedente = pd.read_csv(REVIEW_CSV)

    chiavi = ["recording_id", "canale", "speaker"]
    colonne_manual = [
        "ruolo_manual", "confidenza_manual", "note_manual"
    ]

    manuali = revisione_precedente[
        chiavi + colonne_manual
    ].drop_duplicates(chiavi)

    revisione = revisione_nuova.drop(
        columns=colonne_manual
    ).merge(
        manuali,
        on=chiavi,
        how="left"
    )

    for col in colonne_manual:
        revisione[col] = revisione[col].fillna("")
else:
    revisione = revisione_nuova

ordine_priorita = {
    "alta: più di 2 speaker": 0,
    "alta: meno di 2 speaker": 1,
    "media: canali discordanti": 2,
    "ordinaria": 3,
}

revisione["_ordine"] = (
    revisione["priorita_revisione"]
    .map(ordine_priorita)
    .fillna(99)
)

revisione = revisione.sort_values(
    [
        "_ordine",
        "revisione_obbligatoria",
        "recording_id",
        "durata_totale_secondi",
    ],
    ascending=[True, False, True, False]
).drop(columns="_ordine").reset_index(drop=True)

revisione.to_csv(REVIEW_CSV, index=False)

print("File di revisione:", REVIEW_CSV)
print("Righe:", len(revisione))
print(
    "Revisione obbligatoria:",
    int(revisione["revisione_obbligatoria"].sum())
)


File di revisione: c:\Users\acer\Desktop\ProgettoTesi\risultati\selezione_canale\mappa_ruoli_speaker_da_revisionare.csv
Righe: 197
Revisione obbligatoria: 57


## 7. Visualizzazione di una registrazione

La funzione mostra, per tutti gli speaker della registrazione:

- trascrizione;
- durata e quota di parlato;
- ruolo suggerito;
- anteprima audio.


In [13]:
def mostra_registrazione(recording_id):
    sottoinsieme = revisione[
        revisione["recording_id"].astype(str)
        == str(recording_id)
    ].copy()

    if sottoinsieme.empty:
        print("Registrazione non trovata.")
        return

    display(
        sottoinsieme[
            [
                "speaker",
                "durata_totale_secondi",
                "quota_parlato_nella_registrazione",
                "trascrizione",
                "ruolo_suggerito",
                "confidenza_suggerimento",
                "ruolo_manual",
                "note_manual",
            ]
        ]
    )

    for _, riga in sottoinsieme.iterrows():
        display(
            HTML(
                f"<h4>{riga['speaker']}</h4>"
                f"<p><b>Suggerimento:</b> "
                f"{riga['ruolo_suggerito']} "
                f"({riga['confidenza_suggerimento']})</p>"
                f"<p><b>Testo:</b> "
                f"{riga['trascrizione']}</p>"
            )
        )

        percorso = Path(str(riga["percorso_anteprima"]))
        if percorso.exists():
            display(Audio(filename=str(percorso)))
        else:
            print("Anteprima non trovata:", percorso)


# Esempio:
# mostra_registrazione("ID112.StudioRuggi")


## 8. Revisione interattiva

Il pannello salva il lavoro nel CSV dopo ogni registrazione.

Per le registrazioni con 3 o 4 speaker è possibile assegnare `paziente` a più speaker quando Pyannote ha frammentato la stessa voce.


In [14]:
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError(
        "ipywidgets non è installato. "
        "Eseguire: %pip install ipywidgets"
    ) from exc


RUOLI = [
    "",
    "paziente",
    "intervistatore",
    "voce_registrata",
    "terza_persona",
    "rumore",
    "incerto",
]

CONFIDENZE = ["", "alta", "media", "bassa"]

recording_ids = revisione["recording_id"].drop_duplicates().tolist()

selettore_registrazione = widgets.Dropdown(
    options=recording_ids,
    description="Registrazione:",
    layout=widgets.Layout(width="650px"),
)

output = widgets.Output()
messaggio = widgets.HTML()
pulsante_salva = widgets.Button(
    description="Salva",
    button_style="success",
)
pulsante_successivo = widgets.Button(
    description="Salva e successivo",
    button_style="info",
)

widget_ruoli = {}
widget_confidenze = {}
widget_note = {}


def valore_ruolo_iniziale(riga):
    valore_manual = str(riga.get("ruolo_manual", "") or "").strip()
    if valore_manual in RUOLI:
        return valore_manual

    suggerito = str(riga.get("ruolo_suggerito", "") or "")
    if suggerito == "paziente":
        return "paziente"

    return ""


def render_registrazione(*_):
    global widget_ruoli, widget_confidenze, widget_note

    widget_ruoli = {}
    widget_confidenze = {}
    widget_note = {}

    recording_id = selettore_registrazione.value
    sottoinsieme = revisione[
        revisione["recording_id"] == recording_id
    ]

    with output:
        output.clear_output(wait=True)

        display(
            HTML(
                f"<h3>{recording_id}</h3>"
                f"<p>Speaker: {len(sottoinsieme)} — "
                f"Priorità: "
                f"{sottoinsieme['priorita_revisione'].iloc[0]}</p>"
            )
        )

        for indice, riga in sottoinsieme.iterrows():
            display(
                HTML(
                    f"<hr><h4>{riga['speaker']}</h4>"
                    f"<p><b>Durata:</b> "
                    f"{riga['durata_totale_secondi']:.1f} s "
                    f"({100*riga['quota_parlato_nella_registrazione']:.1f}%)"
                    f"</p>"
                    f"<p><b>Trascrizione:</b> "
                    f"{riga['trascrizione']}</p>"
                    f"<p><b>Suggerimento:</b> "
                    f"{riga['ruolo_suggerito']} "
                    f"— confidenza "
                    f"{riga['confidenza_suggerimento']}</p>"
                )
            )

            percorso = Path(str(riga["percorso_anteprima"]))
            if percorso.exists():
                display(Audio(filename=str(percorso)))

            ruolo = widgets.Dropdown(
                options=RUOLI,
                value=valore_ruolo_iniziale(riga),
                description="Ruolo:",
                layout=widgets.Layout(width="450px"),
            )

            confidenza_corrente = str(
                riga.get("confidenza_manual", "") or ""
            ).strip()
            if confidenza_corrente not in CONFIDENZE:
                confidenza_corrente = ""

            confidenza = widgets.Dropdown(
                options=CONFIDENZE,
                value=confidenza_corrente,
                description="Confidenza:",
                layout=widgets.Layout(width="350px"),
            )

            note = widgets.Text(
                value=str(riga.get("note_manual", "") or ""),
                description="Note:",
                layout=widgets.Layout(width="700px"),
            )

            widget_ruoli[indice] = ruolo
            widget_confidenze[indice] = confidenza
            widget_note[indice] = note

            display(ruolo, confidenza, note)


def salva_corrente():
    for indice, widget in widget_ruoli.items():
        revisione.at[indice, "ruolo_manual"] = widget.value
        revisione.at[indice, "confidenza_manual"] = (
            widget_confidenze[indice].value
        )
        revisione.at[indice, "note_manual"] = (
            widget_note[indice].value
        )

    revisione.to_csv(REVIEW_CSV, index=False)

    registrazione = selettore_registrazione.value
    messaggio.value = (
        f"<span style='color:green'>Salvata: "
        f"{registrazione}</span>"
    )


def on_salva(_):
    salva_corrente()


def on_salva_successivo(_):
    salva_corrente()

    posizione = recording_ids.index(
        selettore_registrazione.value
    )

    if posizione < len(recording_ids) - 1:
        selettore_registrazione.value = (
            recording_ids[posizione + 1]
        )


selettore_registrazione.observe(
    render_registrazione,
    names="value"
)
pulsante_salva.on_click(on_salva)
pulsante_successivo.on_click(on_salva_successivo)

display(
    selettore_registrazione,
    widgets.HBox([pulsante_salva, pulsante_successivo]),
    messaggio,
    output,
)

render_registrazione()


Dropdown(description='Registrazione:', layout=Layout(width='650px'), options=('ID112.StudioRuggi', 'ID118.Stud…

HTML(value='')

Output()

## 9. Stato di avanzamento


In [15]:
revisione_aggiornata = pd.read_csv(REVIEW_CSV)
revisione_aggiornata["ruolo_manual"] = (
    revisione_aggiornata["ruolo_manual"]
    .fillna("")
    .astype(str)
    .str.strip()
)

speaker_compilati = (
    revisione_aggiornata["ruolo_manual"] != ""
).sum()

registrazioni_complete = (
    revisione_aggiornata
    .assign(
        compilato=revisione_aggiornata["ruolo_manual"] != ""
    )
    .groupby("recording_id")["compilato"]
    .all()
    .sum()
)

print(
    f"Speaker compilati: {speaker_compilati}/"
    f"{len(revisione_aggiornata)}"
)
print(
    f"Registrazioni complete: {registrazioni_complete}/"
    f"{revisione_aggiornata['recording_id'].nunique()}"
)


Speaker compilati: 6/197
Registrazioni complete: 1/93


## 10. Validazione ed esportazione finale

La mappa finale viene esportata soltanto quando:

- tutti gli speaker hanno un ruolo manuale;
- ogni registrazione contiene almeno uno speaker etichettato come `paziente`.

Più speaker possono essere etichettati come paziente nei casi di frammentazione della stessa voce.


In [16]:
revisione_finale = pd.read_csv(REVIEW_CSV)

for colonna in [
    "ruolo_manual",
    "confidenza_manual",
    "note_manual",
]:
    revisione_finale[colonna] = (
        revisione_finale[colonna]
        .fillna("")
        .astype(str)
        .str.strip()
    )

ruoli_mancanti = revisione_finale[
    revisione_finale["ruolo_manual"] == ""
]

registrazioni_senza_paziente = []

for recording_id, gruppo in revisione_finale.groupby(
    "recording_id"
):
    if not (gruppo["ruolo_manual"] == "paziente").any():
        registrazioni_senza_paziente.append(recording_id)

print("Ruoli mancanti:", len(ruoli_mancanti))
print(
    "Registrazioni senza paziente:",
    len(registrazioni_senza_paziente)
)

if not ruoli_mancanti.empty:
    print("\nPrime righe senza ruolo:")
    display(
        ruoli_mancanti[
            ["recording_id", "speaker", "priorita_revisione"]
        ].head(20)
    )

if registrazioni_senza_paziente:
    print("\nRegistrazioni senza paziente:")
    print(registrazioni_senza_paziente[:30])

if (
    ruoli_mancanti.empty
    and not registrazioni_senza_paziente
):
    colonne_finali = [
        "recording_id", "canale", "speaker",
        "ruolo_manual", "confidenza_manual",
        "note_manual", "trascrizione",
        "durata_totale_secondi",
        "quota_parlato_nella_registrazione",
    ]

    revisione_finale[colonne_finali].to_csv(
        FINAL_CSV,
        index=False
    )

    print("\nMappa finale salvata in:", FINAL_CSV)
else:
    print(
        "\nEsportazione finale non eseguita: "
        "completare prima la revisione."
    )


Ruoli mancanti: 191
Registrazioni senza paziente: 91

Prime righe senza ruolo:


,recording_id,speaker,priorita_revisione
2,ID112.StudioRuggi,SPEAKER_02,alta: più di 2 speaker
3,ID118.StudioRuggi,SPEAKER_00,alta: più di 2 speaker
4,ID118.StudioRuggi,SPEAKER_01,alta: più di 2 speaker
9,ID155.StudioRuggi,SPEAKER_01,alta: più di 2 speaker
10,ID155.StudioRuggi,SPEAKER_02,alta: più di 2 speaker
11,ID155.StudioRuggi,SPEAKER_00,alta: più di 2 speaker
12,ID30.StudioRuggi,SPEAKER_03,alta: più di 2 speaker
13,ID30.StudioRuggi,SPEAKER_02,alta: più di 2 speaker
14,ID30.StudioRuggi,SPEAKER_00,alta: più di 2 speaker
15,ID30.StudioRuggi,SPEAKER_01,alta: più di 2 speaker



Registrazioni senza paziente:
['ID1.StudioRuggi-006', 'ID10.StudioRuggi', 'ID101.StudioRuggi', 'ID103.StudioRuggi', 'ID104.StudioRuggi', 'ID108.StudioRuggi', 'ID11.StudioRuggi', 'ID114.StudioRuggi', 'ID116.StudioRuggi', 'ID117.StudioRuggi', 'ID118.StudioRuggi', 'ID12.StudioRuggi', 'ID122.StudioRuggi', 'ID126.StudioRuggi', 'ID129.StudioRuggi', 'ID13.StudioRuggi-001', 'ID130.StudioRuggi', 'ID131.StudioRuggi', 'ID136.StudioRuggi', 'ID14.StudioRuggi', 'ID140.StudioRuggi', 'ID145.StudioRuggi', 'ID149.StudioRuggi', 'ID15.StudioRuggi', 'ID154.StudioRuggi', 'ID155.StudioRuggi', 'ID156.StudioRuggi', 'ID157.StudioRuggi', 'ID16.StudioRuggi', 'ID16.StudioRuggi(1)']

Esportazione finale non eseguita: completare prima la revisione.
